# ETCCDI fixed-threshold indices: E3SMv3

The eighteen fixed-threshold ETCCDI indices for E3SMv3 under SSP2-4.5, G6-1.5K-SAI and G6-1.5K-HiLLA, one file per index and member, every year each run has from 2015 to 2084.

The computation is `build_indices.build`, the same code as the command line `python build_indices.py`, with each index defined once in `multimodel_etccdi.INDEX_REGISTRY`. The twelve percentile indices come from the same script; add them to `INDICES` below, or run the script.

Grid: regridded, 180 lat x 360 lon. Calendar: noleap.

## Sources

| Scenario | Members | Source on the bucket |
|---|---|---|
| SSP2-4.5 | r1 to r3 (v3.LR.ssp245_0101, 0151, 0201) | `fafrifa/E3SMv3_regridded/<case>/merged_<VAR>_<y0>_<y1>.nc` |
| G6-1.5K-SAI | r1 to r3 (001 to 003) | same, case `G6-1.5K-SAI_<member>` |
| G6-1.5K-HiLLA | r1 to r3 (v3.LR.ssp245.g6_hilla.sai.0101, 0151, 0201) | same |

Hub only: E3SMv3 is not on GLADE. Each member is assembled day by day from the regridded merged windows on the bucket. EAM stamps a daily mean at the end of its day and the merged files carry no `time_bnds`, so every stamp is moved back 12 hours to the middle of its day.

E3SMv3 daily maximum and minimum temperature (TREFHTMX, TREFHTMN) equal the daily mean in this archive, so the eight tasmax and tasmin indices are not computed. That leaves the eight precipitation indices and, where `merged_TREFHT_*` windows exist, GSL and GDD.

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())          # the repository root
import build_indices as bi

MODEL = 'E3SM'
SITE = os.environ.get('ETCCDI_SITE', 'hub')   # `'hub'` (this model is not on GLADE)
OUT_ROOT = os.environ.get('ETCCDI_OUT_ROOT', 'etccdi_output')
INDICES = list(bi.FIXED_INDICES)        # the eighteen below
SCENARIOS = ['SSP245', 'HiLLA', 'SAI']
MEMBERS = None                          # every member
LAT_RANGE = None                        # e.g. (40, 45) for a quick band
FULL_GRID = True                        # the globe, in latitude bands
OUTPUT = 'per-index'                    # or 'merged', 'both'
WORKERS = 3

if LAT_RANGE:
    FULL_GRID = False
print(MODEL, SITE, OUT_ROOT, len(INDICES), 'indices')

## Before anything is computed

The preflight reads one year of one member, prints the members each scenario will write, and checks the output root. It writes nothing.

In [ ]:
bi.build(MODEL, OUT_ROOT, site=SITE, indices=INDICES,
         scenarios=SCENARIOS, members=MEMBERS, lat_range=LAT_RANGE,
         output=OUTPUT, preflight=True)

## The indices

| Variable | Index | Name | Definition | Unit |
|---|---|---|---|---|
| tasmax | SU | Summer days | Days with Tx > 25 C | d |
| tasmax | ID | Ice days | Days with Tx < 0 C | d |
| tasmax | TXX | Warmest day | Annual maximum of Tx | K |
| tasmax | TXN | Coldest day | Annual minimum of Tx | K |
| tasmin | FD | Frost days | Days with Tn < 0 C | d |
| tasmin | TN | Tropical nights | Days with Tn > 20 C | d |
| tasmin | TNX | Warmest night | Annual maximum of Tn | K |
| tasmin | TNN | Coldest night | Annual minimum of Tn | K |
| pr | PRCPTOT | Total precipitation | Annual sum | mm |
| pr | SDII | Simple daily intensity | Mean precipitation on days >= 1 mm | mm d-1 |
| pr | CDD | Consecutive dry days | Longest run below 1 mm/day | d |
| pr | CWD | Consecutive wet days | Longest run at or above 1 mm/day | d |
| pr | R10MM | Heavy precipitation days | Days >= 10 mm/day | d |
| pr | R20MM | Very heavy precipitation days | Days >= 20 mm/day | d |
| pr | RX1D | Wettest day | Annual maximum daily precipitation | mm d-1 |
| pr | RX5D | Wettest five days | Annual maximum five-day total | mm |
| tas | GSL | Growing season length | ETCCDI definition, both hemispheres | d |
| tas | GDD | Growing degree days | Sum of (tas - 4 C) over days above 4 C | d K |

The four annual temperature extremes are in kelvin: xclim returns the units of
its input. Subtract 273.15 for Celsius when plotting. TN is tropical nights,
which ETCCDI calls TR; the name follows the ReflectiveCloud/etccdi-indices zarr
store. CDD and CWD are measured within each year, so a spell crossing New Year
is cut at the boundary, while RX5D's five-day window reaches back into the
previous year. Both are xclim defaults. A year with fewer than 360 days is
written as NaN.

TXX, TXN, TNX, TNN and the six fixed-threshold counts carry each model's biases
directly; nothing cancels them as a percentile threshold does. Comparisons
between models are safest over the interior of large land masses: within a grid
cell or two of a coast the models' land-sea masks differ and the diurnal range
follows the land fraction.

In [ ]:
bi.build(MODEL, OUT_ROOT, site=SITE, indices=INDICES,
         scenarios=SCENARIOS, members=MEMBERS, lat_range=LAT_RANGE,
         full_grid=FULL_GRID, output=OUTPUT, workers=WORKERS)

## Reading the output

Files follow `<MODEL>/<SCENARIO>/<member>/<INDEX>_<MODEL>_<SCENARIO>_<member>_<first>-<last>.nc`. A rerun skips files that exist; pass `overwrite=True` to replace them.

In [ ]:
import glob
import xarray as xr
if not OUT_ROOT.startswith('s3://'):
    files = sorted(glob.glob(f'{OUT_ROOT}/*/*/*/*.nc'))
    print(len(files), 'files')
    if files:
        print(xr.open_dataset(files[0], decode_timedelta=False))